### test different prediction tree based models

In [19]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn import model_selection
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import joblib

In [20]:
df = pd.read_csv("datasets/final_ds.csv")
target_cols = [f"NCAR{i}" for i in range(1,21)]
subset = target_cols + ['stage', 'date']
df_clean = df.dropna(subset=subset).reset_index(drop=True)

In [21]:
df_clean['date'] = pd.to_datetime(df_clean['date'])

In [22]:
train_mask = (df_clean['date'] >= '2025-01-01') & (df_clean['date'] <= '2025-11-30')
test_mask = (df_clean['date'] >= '2026-01-01') & (df_clean['date'] <= '2026-06-30')

In [23]:
features = ['polarity',
            'drug_count',
            'Total Revenue', 
            'Operating Revenue', 
            'Cash And Cash Equivalents', 
            'Capital Expenditure', 
            'Stock_Trend_30d', 
            'Stock_Trend_60d',
            'NBI_Trend_30d', 
            'NBI_Trend_60d',
            'isDelisted'
            ]

In [24]:
X_train = df_clean.loc[train_mask, features]
X_test = df_clean.loc[test_mask, features]

# X_train['stage'] = X_train['stage'].astype('category')
# X_test['stage'] = X_test['stage'].astype('category')

X_train['isDelisted'] = X_train['isDelisted'].astype('int')
X_test['isDelisted'] = X_test['isDelisted'].astype('int')

cont_vars = [c for c in features if (c != 'stage' and c!= 'isDelisted')]
X_train[cont_vars] = X_train[cont_vars].apply(pd.to_numeric)
X_test[cont_vars] = X_test[cont_vars].apply(pd.to_numeric)

print(f"Train observation count: {len(X_train)}")
print(f"Test observation count: {len(X_test)}")

Train observation count: 2316
Test observation count: 1477


In [25]:
seed = 1234
candidate_models = [("HGB", HistGradientBoostingRegressor(random_state = seed)),
                    ("DTR", DecisionTreeRegressor(random_state = seed)),
                    ("RFR", RandomForestRegressor(n_estimators = 100, random_state = seed, n_jobs=1)),
                    ("XGB", xgb.XGBRegressor(
                        random_state = seed,
                        n_jobs = 1,
                        tree_method = "hist",
                        learning_rate = 0.05,
                        enable_categorical = True,
                        missing = np.nan))
                    ]

In [ ]:
trained_models = {name: {} for name, _ in candidate_models}
eval_metrics = []

for dayX in range(1,21):
    target = f'NCAR{dayX}'
    y_train = df_clean.loc[train_mask, target]
    y_test = df_clean.loc[test_mask, target]
    y_train = y_train.apply(pd.to_numeric)
    y_test = y_test.apply(pd.to_numeric)

    for name, model in candidate_models:
        if name == "XGB":
            model.fit(X_train,
                    y_train,
                    eval_set = [(X_test, y_test)],
                    verbose= False
                    )
        else:
            model.fit(X_train, y_train)

        # model.save_model("model.json")

        trained_models[name][dayX] = model

        y_pred = model.predict(X_test)

        # model evaluation

        rank_ic, p_val = spearmanr(y_test, y_pred)

        eval_metrics.append({
            "Model": name,
            "Day": dayX,
            "Directional Accuracy": np.mean(np.sign(y_pred) == np.sign(y_test)),
            "Rank IC": rank_ic, 
            "p_val": p_val,
            "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
            "MAE": mean_absolute_error(y_test, y_pred),
            "R2": r2_score(y_test, y_pred)
        })

In [27]:
eval_metrics = pd.DataFrame(eval_metrics)
print(eval_metrics)
eval_metrics.to_csv("eval.csv", index = False)


   Model  Day  Directional Accuracy   Rank IC         p_val      RMSE  \
0    HGB    1              0.467163  0.044958  8.412982e-02  0.372260   
1    DTR    1              0.532160  0.053116  4.124539e-02  0.533469   
2    RFR    1              0.501016  0.055015  3.450352e-02  0.367665   
3    XGB    1              0.498307  0.079435  2.249988e-03  0.377244   
4    HGB    2              0.454976 -0.020305  4.355315e-01  1.058174   
..   ...  ...                   ...       ...           ...       ...   
75   XGB   19              0.549086  0.250138  1.660191e-22  2.317072   
76   HGB   20              0.488152  0.139938  6.660837e-08  2.096089   
77   DTR   20              0.521327  0.066310  1.080197e-02  2.834064   
78   RFR   20              0.537576  0.176501  8.420289e-12  2.230994   
79   XGB   20              0.560596  0.266096  2.327246e-25  2.402356   

         MAE        R2  
0   0.106485 -0.096636  
1   0.132935 -1.252097  
2   0.089682 -0.069728  
3   0.092570 -0.126198 